# Baseline Model

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import models
from torchvision.models import ResNet18_Weights

In [2]:
df = pd.read_csv("../data/all_tracks_clean.csv")

df.head()

,genre,id,title,artist,album,duration_sec,rank,explicit
0,Pop,3503857201,Man I Need,Olivia Dean,Man I Need,184,991622,False
1,Pop,3579685911,The Fate of Ophelia,Taylor Swift,The Life of a Showgirl,226,887228,False
2,Pop,3210709941,Ordinary,Alex Warren,Ordinary,186,990241,False
3,Pop,3454558671,YUKON,Justin Bieber,SWAG,164,881380,False
4,Pop,3763842212,I Just Might,Bruno Mars,I Just Might,212,995346,False


In [3]:
df.shape

(7580, 8)

In [4]:
image_data = np.load("../data/spectrogram_tensors.npz")

X = image_data["X"]
indices = image_data["indices"]

In [5]:
# Log-transform rank to reduce skew before scaling
y = np.log1p(df.iloc[indices]["rank"].values)

In [6]:
X.shape, y.shape

((7549, 1, 128, 128), (7549,))

In [7]:
from sklearn.model_selection import train_test_split

#Train/Test/Val Split (70/15/15)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=6500)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=6500)

In [8]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
y_train = scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
y_val   = scaler.transform(y_val.reshape(-1, 1)).flatten()
y_test  = scaler.transform(y_test.reshape(-1, 1)).flatten()

In [9]:
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(5284, 1, 128, 128) (5284,)
(1132, 1, 128, 128) (1132,)
(1133, 1, 128, 128) (1133,)


In [10]:
def to_tensor_dataset(X, y):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    return TensorDataset(X_t, y_t)

train_ds = to_tensor_dataset(X_train, y_train)
val_ds   = to_tensor_dataset(X_val,   y_val)
test_ds  = to_tensor_dataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32)
test_loader  = DataLoader(test_ds,  batch_size=32)

In [11]:
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.fc.in_features, 1)
)

In [12]:
# Freeze all backbone params — only train the fc head initially
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

In [13]:
device = torch.device("mps")
model  = model.to(device)

In [14]:
criterion = nn.HuberLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

In [15]:
def spec_augment(x, freq_mask_param=20, time_mask_param=20):
    """Apply random frequency and time masking to a batch of spectrograms."""
    B, C, F, T = x.shape
    # Frequency masking
    f = torch.randint(0, freq_mask_param, (1,)).item()
    f0 = torch.randint(0, max(1, F - f), (1,)).item()
    x[:, :, f0:f0+f, :] = 0
    # Time masking
    t = torch.randint(0, time_mask_param, (1,)).item()
    t0 = torch.randint(0, max(1, T - t), (1,)).item()
    x[:, :, :, t0:t0+t] = 0
    return x

def train_epoch(model, loader):
    model.train()
    total_loss, total_mae = 0, 0
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device, dtype=torch.float32)
        y_batch = y_batch.to(device, dtype=torch.float32)
        X_batch = spec_augment(X_batch)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss  = criterion(preds.squeeze(1), y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_mae  += torch.abs(preds.squeeze(1) - y_batch).sum().item()
    return total_loss / len(loader), total_mae / len(loader.dataset)

def eval_epoch(model, loader):
    model.eval()
    total_loss, total_mae = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device, dtype=torch.float32)
            y_batch = y_batch.to(device, dtype=torch.float32)
            preds = model(X_batch)
            total_loss += criterion(preds.squeeze(1), y_batch).item()
            total_mae  += torch.abs(preds.squeeze(1) - y_batch).sum().item()
    return total_loss / len(loader), total_mae / len(loader.dataset)

In [16]:
EPOCHS        = 100
WARMUP_EPOCHS = 5
PATIENCE      = 5

In [ ]:
# Phase 1: warm up — only the fc head is trainable
print("=== Phase 1: Warming up head ===")
for epoch in range(WARMUP_EPOCHS):
    train_loss, train_mae = train_epoch(model, train_loader)
    val_loss,   val_mae   = eval_epoch(model, val_loader)
    scheduler.step(val_loss)
    print(f"[Warmup] Epoch {epoch+1:02d} | "
          f"Train loss: {train_loss:.4f}  MAE: {train_mae:.4f} | "
          f"Val loss: {val_loss:.4f}  MAE: {val_mae:.4f}")

# Phase 2: unfreeze last ResNet block + fc, use lower LR
print("\n=== Phase 2: Fine-tuning layer4 + head ===")
for param in model.layer4.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

best_val_loss   = float("inf")
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_mae = train_epoch(model, train_loader)
    val_loss,   val_mae   = eval_epoch(model, val_loader)
    scheduler.step(val_loss)

    print(f"[Finetune] Epoch {epoch+1:02d} | "
          f"Train loss: {train_loss:.4f}  MAE: {train_mae:.4f} | "
          f"Val loss: {val_loss:.4f}  MAE: {val_mae:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping triggered")
            break

=== Phase 1: Warming up head ===
[Warmup] Epoch 01 | Train loss: 0.1216  MAE: 0.3813 | Val loss: 0.0266  MAE: 0.1910
[Warmup] Epoch 02 | Train loss: 0.0286  MAE: 0.1875 | Val loss: 0.0315  MAE: 0.2181
[Warmup] Epoch 03 | Train loss: 0.0187  MAE: 0.1525 | Val loss: 0.0476  MAE: 0.2811
[Warmup] Epoch 04 | Train loss: 0.0181  MAE: 0.1509 | Val loss: 0.0192  MAE: 0.1647
[Warmup] Epoch 05 | Train loss: 0.0172  MAE: 0.1457 | Val loss: 0.0273  MAE: 0.2052

=== Phase 2: Fine-tuning layer4 + head ===
[Finetune] Epoch 01 | Train loss: 0.0124  MAE: 0.1216 | Val loss: 0.0083  MAE: 0.0990
[Finetune] Epoch 02 | Train loss: 0.0099  MAE: 0.1077 | Val loss: 0.0076  MAE: 0.0943
[Finetune] Epoch 03 | Train loss: 0.0093  MAE: 0.1035 | Val loss: 0.0078  MAE: 0.0907
[Finetune] Epoch 04 | Train loss: 0.0085  MAE: 0.0989 | Val loss: 0.0075  MAE: 0.0971
[Finetune] Epoch 05 | Train loss: 0.0084  MAE: 0.0965 | Val loss: 0.0088  MAE: 0.1108
[Finetune] Epoch 06 | Train loss: 0.0082  MAE: 0.0965 | Val loss: 0.0072 

In [ ]:
model.load_state_dict(torch.load("best_model.pt"))
test_loss, test_mae = eval_epoch(model, test_loader)
print(f"Test loss: {test_loss:.4f} | Test MAE (log-normalized): {test_mae:.4f}")

# Collect all predictions and targets to compute original-scale MAE
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device, dtype=torch.float32)
        preds = model(X_batch).squeeze(1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y_batch.numpy())

all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

# Invert MinMaxScaler → log(rank), then expm1 → original rank
preds_rank   = np.expm1(scaler.inverse_transform(all_preds.reshape(-1, 1)).flatten())
targets_rank = np.expm1(scaler.inverse_transform(all_targets.reshape(-1, 1)).flatten())

test_mae_original = np.mean(np.abs(preds_rank - targets_rank))
print(f"Test MAE (original scale): {test_mae_original:.1f}")

Test loss: 0.0076 | Test MAE (log-normalized): 0.0904
Test MAE (original scale): 208677.6


In [19]:
print("count:", y.size)
print("mean:", np.mean(y))
print("std:", np.std(y))
print("min:", np.min(y))
print("25%:", np.percentile(y, 25))
print("median:", np.median(y))
print("75%:", np.percentile(y, 75))
print("max:", np.max(y))

count: 7549
mean: 12.474203964181203
std: 1.1785914672337927
min: 5.159055299214529
25%: 12.004293002079097
median: 12.870982375478876
75%: 13.301289175129103
max: 13.81522251648431
